# Trajectory Pilot — 未来 10s ranktrace 回归

**任务**：给定过去 20s 多模态窗口，预测未来 10s ranktrace（51 点 @ 5Hz）。

| Field | Value |
|---|---|
| Sweep | `configs/sweeps/trajectory_pilot.yaml` |
| Task | `eft_trajectory_pilot`（未来 10s 回归，out_dim=51） |
| Head | `regression` |
| Loss | `mse` |
| Early stopping | `val_mse`（min） |
| Split | `cross_subject` (session_tvt.json) |
| Seeds | `[0]` |
| Modality ablation | `[video]` / `[video,telem]` / `[video,km,telem]`（共 3 runs） |
| Fusion | EFT（4 层，对齐 phase1） |
| LR / batch / epochs / patience | `5e-5 / 256 / 40 / 8`（对齐 phase1_ablation） |

**参考阈值**：旧 3-class 范式 cross_subject state ceiling ≈ 0.44 macro-F1。本任务换评估范式（MSE + event-locked），不直接比 F1，而是看：
- val MSE / CCC 收敛情况
- 跨模态 ablation 单调性（V < VT < VKT）
- 后续 post-hoc 用 `src/metrics/event_locked.py` 评估事件锁相指标

**产出**：`runs/trajectory_pilot/cross_subject/eft_trajectory_pilot/<modality>/seed0/metrics.json` × 3

In [9]:
# Cell 1: Mount Drive + setup
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyyaml', 'scipy', '-q'])

%cd /content/drive/MyDrive/ProjectExperiment
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ProjectExperiment
NVIDIA L4, 23034 MiB


In [10]:
# Cell 2: Generate events_index.json if missing
import os
EVENTS_PATH = '/content/drive/MyDrive/AmuCS_experiment/labels/events_index.json'
if os.path.exists(EVENTS_PATH):
    print(f'events_index.json already exists: {EVENTS_PATH}')
else:
    print('Building events_index.json ...')
    !python scripts/build_event_index.py \
      --amucs_root "/content/drive/MyDrive/AmuCS/Affective Multimodal Counter-Strike video game dataset (AMuCS) - Public/researchdata/data" \
      --telem_dir /content/drive/MyDrive/AmuCS_experiment/features/aligned/telem \
      --stems_from /content/drive/MyDrive/AmuCS_experiment/labels/labels_arousal_seq.json \
      --out_path /content/drive/MyDrive/AmuCS_experiment/labels/events_index.json

events_index.json already exists: /content/drive/MyDrive/AmuCS_experiment/labels/events_index.json


In [11]:
# Cell 3: Dry run — verify plan = 3 runs, no import errors
!python -u scripts/run_experiment.py \
    --sweep configs/sweeps/trajectory_pilot.yaml \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot \
    --dry_run 2>&1 | tail -30

Sweep plan: 3 runs total, 0 already completed, 3 to run

  [RUN] eft_trajectory_pilot / single_video / seed0
  [RUN] eft_trajectory_pilot / dual_video_telem / seed0
  [RUN] eft_trajectory_pilot / triple_video_km_telem / seed0


In [12]:
# Cell 4: Run 3 modality ablation runs (re-run safe — skips if done)
!python -u scripts/run_experiment.py \
    --sweep configs/sweeps/trajectory_pilot.yaml \
    --workers 1 \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot

Sweep plan: 3 runs total, 0 already completed, 3 to run


[1/3] [cross_subject] eft_trajectory_pilot / single_video / seed0
[AMuCSTrajectoryDataModule] train-set Δσ = 18.4468
torch.compile enabled
Model parameters: 8,950,323 (trainable: 8,950,323)
Train samples: 19123, Val samples: 6259
Run directory: /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_3seed/single_video/2026-04-23_19-30-36__amucs_trajectory__eft__video__seed0
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
W0423 19:30:54.274000 17512 torch/_inductor/utils.py:1679] [9/0] Not enough SMs to use max_autotune_gemm mode
Epoch 1/40 | train_ccc: 0.0011 | train_loss: 1.2250 | train_mse: 1.2249 | val_ccc: 0.0004 | val_mse: 1.5847
Epoch 2/40 | train_ccc: 0.0

In [13]:
# Cell 5: Summarise 3 runs
import json, glob
from pathlib import Path

PATTERN = '/content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/**/metrics.json'
files = sorted(glob.glob(PATTERN, recursive=True))
print(f'Found {len(files)} run(s)')

rows = []
for fp in files:
    m = json.load(open(fp))
    # extract modality tag from path
    parts = Path(fp).parts
    mod_tag = next((p for p in parts if p.startswith(('single_', 'double_', 'triple_', 'video'))), 'unknown')
    rows.append({
        'modality': mod_tag,
        'train_mse': m.get('train_mse'),
        'val_mse': m.get('val_mse'),
        'test_mse': m.get('test_mse'),
        'val_ccc': m.get('val_ccc'),
        'test_ccc': m.get('test_ccc'),
        'best_epoch': m.get('best_epoch'),
    })

# Pretty print
print(f"{'modality':<40} {'tr_mse':>8} {'va_mse':>8} {'te_mse':>8} {'va_ccc':>8} {'te_ccc':>8} {'ep':>4}")
print('-' * 90)
for r in rows:
    def fmt(v, w=8, d=4):
        return f'{v:{w}.{d}f}' if isinstance(v, (int, float)) else f'{str(v):>{w}}'
    print(f"{r['modality']:<40} {fmt(r['train_mse'])} {fmt(r['val_mse'])} {fmt(r['test_mse'])} "
          f"{fmt(r['val_ccc'])} {fmt(r['test_ccc'])} {fmt(r['best_epoch'], 4, 0)}")

print()
print('Sanity checks:')
print('  - val_mse 应单调下降 (V > VT > VKT) 如果模态互补')
print('  - val_mse < 1.0 说明预测偏差 < 1 train_σ（训练 σ 见 runs 日志里的 Δσ）')
print('  - val_ccc > 0 说明预测和 GT 正相关（0 = 瞎猜，1 = 完美）')

Found 3 run(s)
modality                                   tr_mse   va_mse   te_mse   va_ccc   te_ccc   ep
------------------------------------------------------------------------------------------
unknown                                    0.8369   1.6238   2.3793  -0.0002  -0.0081    2
single_video                               0.8308   1.6503   2.7388   0.0073  -0.0093    2
triple_video_km_telem                      0.7604   1.5504   2.2975   0.0176   0.0006    2

Sanity checks:
  - val_mse 应单调下降 (V > VT > VKT) 如果模态互补
  - val_mse < 1.0 说明预测偏差 < 1 train_σ（训练 σ 见 runs 日志里的 Δσ）
  - val_ccc > 0 说明预测和 GT 正相关（0 = 瞎猜，1 = 完美）
